# India Client-Finding Agent — Qwen 2.5 3B on Kaggle free GPU

**Before running, set (right sidebar → Session options):**
1. **Accelerator: GPU T4 x1** (or P100)
2. **Internet: ON** (required to download Qwen + investigate websites)
3. Optional secrets (Add-ons → Secrets): `GOOGLE_MAPS_API_KEY`, `REDDIT_CLIENT_ID`, `REDDIT_CLIENT_SECRET`, `BRAVE_SEARCH_KEY`, `HF_TOKEN`
4. Optional: attach your previous **`icf-state` dataset** (Add Input) to resume from the last checkpoint

Flow: clone repo → install deps → restore checkpoint → load Qwen (4-bit) → discover + qualify a batch → save leads → export Excel → package state for next session.

In [ ]:
# 1) Get the code -----------------------------------------------------------
REPO_URL = "https://github.com/Naserkhan07/soul_exter.git"   # <-- your repo
BRANCH   = "arena/01a02332-soul-exter"

import os, shutil
if os.path.exists("/kaggle/working/project"):
    shutil.rmtree("/kaggle/working/project")
rc = os.system(f"git clone --depth 1 -b {BRANCH} {REPO_URL} /kaggle/working/project")
if rc != 0:
    # Repo private / no internet clone? Upload the repo as a Kaggle Dataset
    # named e.g. 'india-client-finder-code' and copy it instead:
    src = "/kaggle/input/india-client-finder-code"
    assert os.path.exists(src), "git clone failed and no code dataset attached"
    shutil.copytree(src, "/kaggle/working/project")
%cd /kaggle/working/project
!ls

In [ ]:
# 2) Install dependencies ----------------------------------------------------
!pip -q install transformers accelerate bitsandbytes \
                requests beautifulsoup4 lxml openpyxl datasets
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 3) Restore previous state (attach your 'icf-state' dataset as input) -------
import shutil, os
STATE_INPUT = "/kaggle/input/icf-state"   # dataset made from a previous session
if os.path.exists(STATE_INPUT):
    for rel in ("data/leads.db", "data/checkpoints/agent_checkpoint.json"):
        src = os.path.join(STATE_INPUT, rel)
        if os.path.exists(src):
            os.makedirs(os.path.dirname(rel), exist_ok=True)
            shutil.copy(src, rel)
            print("restored", rel)
else:
    print("no previous state attached — starting fresh")

In [ ]:
# 4) Load optional API keys from Kaggle Secrets -------------------------------
import os
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ("GOOGLE_MAPS_API_KEY", "REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET",
                "BRAVE_SEARCH_KEY", "SERPAPI_KEY", "HF_TOKEN"):
        try:
            os.environ[key] = secrets.get_secret(key)
            print("loaded secret:", key)
        except Exception:
            pass
except Exception as e:
    print("secrets unavailable:", e)

os.environ["BRAIN_BACKEND"] = "transformers"
os.environ["QWEN_MODEL_ID"] = "Qwen/Qwen2.5-3B-Instruct"
os.environ["QWEN_LOAD_IN_4BIT"] = "1"

In [ ]:
# 5) Run: smoke test + batch (Qwen decides, leads saved, Excel exported) -----
!python kaggle/run_qwen_batch.py --max 200

In [ ]:
# 6) Package state for the NEXT session ---------------------------------------
# Everything lands in /kaggle/working/icf-state. After the session:
#   - first time: Create Dataset from this folder, name it 'icf-state'
#   - later:      update that dataset with a new version
# Attach it as input next session and the agent resumes automatically.
import shutil, os
out = "/kaggle/working/icf-state"
shutil.rmtree(out, ignore_errors=True)
os.makedirs(out + "/data/checkpoints", exist_ok=True)
os.makedirs(out + "/output", exist_ok=True)
for rel in ("data/leads.db", "data/checkpoints/agent_checkpoint.json"):
    if os.path.exists(rel):
        shutil.copy(rel, os.path.join(out, rel))
if os.path.exists("output/india_leads.xlsx"):
    shutil.copy("output/india_leads.xlsx", out + "/output/india_leads.xlsx")
print("State packaged:")
for root, _, files in os.walk(out):
    for f in files:
        print(" ", os.path.join(root, f))
print("\nDownload output/india_leads.xlsx from the Output tab — that's your deliverable.")